In [5]:
import pickle
import time

t0 = time.perf_counter()

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_with_accidents_new.pkl", 'rb') as f:
    berlin_graph_accidents = pickle.load(f)

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_with_nature_new.pkl", 'rb') as f:
    berlin_graph_nature = pickle.load(f)

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_with_population_new.pkl", 'rb') as f:
    berlin_graph_population = pickle.load(f)

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\all_data.pkl", 'rb') as f:
    all_data = pickle.load(f)

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_geo_com.pkl", 'rb') as f:
    berlin_graph_geo_com = pickle.load(f)

logger.info(f"Daten erfolgreich geladen in {time.perf_counter() - t0:.2f}s.")


2026-06-26 13:40:49,378 - INFO - Daten erfolgreich geladen in 1.82s.


In [9]:
# ============================================================
# BERLIN HAZMAT-CVRP PROTOTYP
# - 1 Depot
# - mehrere Kunden pro Route
# - 2 LKWs
# - Zeitfenster
# - Risiko + Kosten
# - Berliner Netzrelationen
# - echte Fahrzeugtouren
# ============================================================

import pandas as pd
import numpy as np
import pulp as pl
import networkx as nx
import logging
import sys
import pickle
import time
import folium

from folium.plugins import MiniMap, HeatMap
from math import radians, sin, cos, sqrt, atan2

# ============================================================
# LOGGING
# ============================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
    force=True
)
logger = logging.getLogger(__name__)

# ============================================================
# ZEITFUNKTIONEN
# ============================================================
def hhmm_to_min(hhmm):
    h, m = map(int, hhmm.split(":"))
    return 60 * h + m

def min_to_hhmm(minutes):
    minutes = int(round(minutes))
    h = minutes // 60
    m = minutes % 60
    return f"{h:02d}:{m:02d}"

# ============================================================
# GEWICHTUNG
# ============================================================
w_cost = 0.50
w_risk = 0.50

# Beispiele:
# w_cost = 0.90 ; w_risk = 0.10
# w_cost = 0.50 ; w_risk = 0.50
# w_cost = 0.10 ; w_risk = 0.90

# ============================================================
# DATEN LADEN
# ============================================================
t0 = time.perf_counter()

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_with_accidents_new.pkl", "rb") as f:
    berlin_graph_accidents = pickle.load(f)

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_with_nature_new.pkl", "rb") as f:
    berlin_graph_nature = pickle.load(f)

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_with_population_new.pkl", "rb") as f:
    berlin_graph_population = pickle.load(f)

logger.info(f"Daten erfolgreich geladen in {time.perf_counter() - t0:.2f}s.")

# ============================================================
# HILFSFUNKTIONEN
# ============================================================
def haversine_scalar(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def normalize(values_dict, invert=False):
    vals = np.array(list(values_dict.values()), dtype=float)
    if len(vals) == 0:
        return {}
    vmin = vals.min()
    vmax = vals.max()

    if vmax == vmin:
        return {k: 0.0 for k in values_dict}

    out = {}
    for k, v in values_dict.items():
        x = (v - vmin) / (vmax - vmin + 1e-9)
        if invert:
            x = 1.0 - x
        out[k] = float(x)
    return out

def risk_to_color(value, vmin=0.0, vmax=1.0):
    if vmax == vmin:
        x = 0.0
    else:
        x = (value - vmin) / (vmax - vmin)
    x = max(0.0, min(1.0, x))
    r = int(255 * x)
    g = int(255 * (1 - x))
    return f"#{r:02x}{g:02x}00"

# ============================================================
# NETZ AUFBAUEN
# ============================================================
df_nodes = berlin_graph_accidents["nodes"].copy()
df_edges = berlin_graph_accidents["edges"].copy()

node_coords = (
    df_nodes
    .set_index("node")[["lat", "lon"]]
    .apply(tuple, axis=1)
    .to_dict()
)

node_ids = df_nodes["node"].values
node_lats = df_nodes["lat"].values
node_lons = df_nodes["lon"].values

def nearest_node(lat, lon):
    dists = haversine_vectorized(lat, lon, node_lats, node_lons)
    idx = np.argmin(dists)
    return int(node_ids[idx]), float(dists[idx])

logger.info(f"Knoten: {len(df_nodes):,}")
logger.info(f"Kanten: {len(df_edges):,}")

# ============================================================
# RISIKODATEN
# ============================================================
pop_dict = dict(zip(
    zip(berlin_graph_population["edges"]["u"], berlin_graph_population["edges"]["v"]),
    berlin_graph_population["edges"]["pop_per_meter"]
))

acc_dict = dict(zip(
    zip(berlin_graph_accidents["edges"]["u"], berlin_graph_accidents["edges"]["v"]),
    berlin_graph_accidents["edges"]["score"]
))

nat_dict = dict(zip(
    zip(berlin_graph_nature["edges"]["u"], berlin_graph_nature["edges"]["v"]),
    berlin_graph_nature["edges"]["dist_to_nature_m"]
))

pop_norm = normalize(pop_dict)
acc_norm = normalize(acc_dict)
nat_norm = normalize(nat_dict, invert=True)

alpha = 0.4
beta  = 0.4
gamma = 0.2

edge_risk_base = {}

for _, row in df_edges.iterrows():
    e = (int(row["u"]), int(row["v"]))
    pop = pop_norm.get(e, 0.0)
    acc = acc_norm.get(e, 0.0)
    nat = nat_norm.get(e, 0.0)

    consequence = alpha * pop + gamma * nat
    probability = beta * acc
    edge_risk_base[e] = probability * consequence

# ============================================================
# STRASSENGRAPH
# ============================================================
G_road = nx.DiGraph()

for _, row in df_edges.iterrows():
    u = int(row["u"])
    v = int(row["v"])

    if "distance" in df_edges.columns:
        dist_km = float(row["distance"]) / 1000.0
    elif "length" in df_edges.columns:
        dist_km = float(row["length"]) / 1000.0
    else:
        coord_u = node_coords.get(u)
        coord_v = node_coords.get(v)
        if coord_u is None or coord_v is None:
            continue
        dist_km = haversine_scalar(coord_u[0], coord_u[1], coord_v[0], coord_v[1]) / 1000.0

    coords = None
    if u in node_coords and v in node_coords:
        coords = [node_coords[u], node_coords[v]]

    G_road.add_edge(
        u, v,
        distance_km=dist_km,
        accident_score=acc_norm.get((u, v), 0.0),
        population_score=pop_norm.get((u, v), 0.0),
        nature_score=nat_norm.get((u, v), 0.0),
        base_risk=edge_risk_base.get((u, v), 0.0),
        coords=coords
    )

logger.info(f"G_road aufgebaut: {G_road.number_of_nodes():,} Knoten, {G_road.number_of_edges():,} Kanten")

# ============================================================
# HEATMAP-DATEN
# ============================================================
def build_heatmap_data(norm_dict):
    heat_data = []
    for (u, v), norm_val in norm_dict.items():
        coord_u = node_coords.get(u)
        coord_v = node_coords.get(v)
        if coord_u and coord_v and norm_val > 0:
            mid_lat = (coord_u[0] + coord_v[0]) / 2
            mid_lon = (coord_u[1] + coord_v[1]) / 2
            heat_data.append([mid_lat, mid_lon, float(norm_val)])
    return heat_data

pop_heat = build_heatmap_data(pop_norm)
acc_heat = build_heatmap_data(acc_norm)
nat_heat = build_heatmap_data(nat_norm)

# ============================================================
# PFADPROFILE
# ============================================================
PATH_PROFILES = {
    "shortest":   0.0,
    "balanced_1": 0.5,
    "balanced_2": 1.0,
    "balanced_3": 2.0,
    "safest":     4.0
}

AVG_SPEED_KMH = 35.0
path_cache = {}

def relation_metrics(lat1, lon1, lat2, lon2, profile="shortest", loaded=True):
    """
    loaded=True  -> Risiko wirkt
    loaded=False -> Rueckfahrt/leer, kein Risiko
    """
    start_node, start_offset_m = nearest_node(lat1, lon1)
    end_node, end_offset_m = nearest_node(lat2, lon2)

    cache_key = (start_node, end_node, profile, loaded)
    if cache_key in path_cache:
        return path_cache[cache_key]

    H = nx.DiGraph()
    lam = PATH_PROFILES[profile]

    for u, v, data in G_road.edges(data=True):
        dist_km = data["distance_km"]
        risk_val = data["base_risk"] * 100.0 if loaded else 0.0

        H.add_edge(
            u, v,
            distance_km=dist_km,
            risk_val=risk_val,
            path_weight=dist_km + lam * risk_val
        )

    try:
        path_nodes = nx.shortest_path(H, source=start_node, target=end_node, weight="path_weight")
    except nx.NetworkXNoPath:
        result = {
            "dist_km": 1e6,
            "time_min": 1e6,
            "risk": 1e6,
            "edge_list": [],
            "start_node": start_node,
            "end_node": end_node
        }
        path_cache[cache_key] = result
        return result

    edge_list = list(zip(path_nodes[:-1], path_nodes[1:]))

    dist_km = 0.0
    risk_sum = 0.0

    for e in edge_list:
        ed = H[e[0]][e[1]]
        dist_km += ed["distance_km"]
        risk_sum += ed["risk_val"]

    dist_km += (start_offset_m + end_offset_m) / 1000.0
    time_min = 60.0 * dist_km / AVG_SPEED_KMH

    result = {
        "dist_km": dist_km,
        "time_min": time_min,
        "risk": risk_sum,
        "edge_list": edge_list,
        "start_node": start_node,
        "end_node": end_node
    }
    path_cache[cache_key] = result
    return result

# ============================================================
# DEPOT / KUNDEN / FAHRZEUGE
# ============================================================
depot_lat = 52.5200
depot_lon = 13.4050

customers = pd.DataFrame([
    {
        "customer_id": 1,
        "lat": 52.4986,
        "lon": 13.4030,
        "demand_kg": 6000,
        "service_time_min": 45,
        "earliest_min": hhmm_to_min("08:30"),
        "latest_min": hhmm_to_min("12:00")
    },
    {
        "customer_id": 2,
        "lat": 52.5163,
        "lon": 13.3041,
        "demand_kg": 5000,
        "service_time_min": 45,
        "earliest_min": hhmm_to_min("09:00"),
        "latest_min": hhmm_to_min("12:30")
    },
    {
        "customer_id": 3,
        "lat": 52.5695,
        "lon": 13.4010,
        "demand_kg": 4000,
        "service_time_min": 45,
        "earliest_min": hhmm_to_min("09:30"),
        "latest_min": hhmm_to_min("13:00")
    },
    {
        "customer_id": 4,
        "lat": 52.4800,
        "lon": 13.4500,
        "demand_kg": 3000,
        "service_time_min": 45,
        "earliest_min": hhmm_to_min("08:45"),
        "latest_min": hhmm_to_min("13:15")
    },
    {
        "customer_id": 5,
        "lat": 52.4700,
        "lon": 13.3300,
        "demand_kg": 4500,
        "service_time_min": 45,
        "earliest_min": hhmm_to_min("10:00"),
        "latest_min": hhmm_to_min("14:00")
    },
    {
        "customer_id": 6,
        "lat": 52.5600,
        "lon": 13.2800,
        "demand_kg": 3500,
        "service_time_min": 45,
        "earliest_min": hhmm_to_min("10:15"),
        "latest_min": hhmm_to_min("14:30")
    }
])

vehicles = pd.DataFrame([
    {
        "vehicle_id": "Truck1",
        "capacity_kg": 16000,
        "fixed_cost": 220,
        "variable_cost_per_km": 1.20,
        "energy_kwh_per_km": 1.50,
        "energy_price": 0.35,
        "shift_start_min": hhmm_to_min("08:00"),
        "shift_end_min": hhmm_to_min("15:00")
    },
    {
        "vehicle_id": "Truck2",
        "capacity_kg": 12000,
        "fixed_cost": 180,
        "variable_cost_per_km": 1.00,
        "energy_kwh_per_km": 1.20,
        "energy_price": 0.35,
        "shift_start_min": hhmm_to_min("08:00"),
        "shift_end_min": hhmm_to_min("15:00")
    }
])

C = customers["customer_id"].tolist()
V = vehicles["vehicle_id"].tolist()
N = [0] + C  # 0 = Depot

Demand = dict(zip(customers["customer_id"], customers["demand_kg"]))
Service = dict(zip(customers["customer_id"], customers["service_time_min"]))
Earliest = dict(zip(customers["customer_id"], customers["earliest_min"]))
Latest = dict(zip(customers["customer_id"], customers["latest_min"]))

Cap = dict(zip(vehicles["vehicle_id"], vehicles["capacity_kg"]))
FixCost = dict(zip(vehicles["vehicle_id"], vehicles["fixed_cost"]))
ShiftStart = dict(zip(vehicles["vehicle_id"], vehicles["shift_start_min"]))
ShiftEnd = dict(zip(vehicles["vehicle_id"], vehicles["shift_end_min"]))
VarCost = dict(zip(vehicles["vehicle_id"], vehicles["variable_cost_per_km"]))
EnergyPerKm = dict(zip(vehicles["vehicle_id"], vehicles["energy_kwh_per_km"]))
EnergyPrice = dict(zip(vehicles["vehicle_id"], vehicles["energy_price"]))

customer_coords = {
    row["customer_id"]: (row["lat"], row["lon"])
    for _, row in customers.iterrows()
}
customer_coords[0] = (depot_lat, depot_lon)

# ============================================================
# RELATIONEN ZWISCHEN DEPOT/KUNDEN
# ============================================================
relation_alts = {}
P_rel = {}

for i in N:
    relation_alts[i] = {}
    P_rel[i] = {}

    for j in N:
        if i == j:
            continue

        lat1, lon1 = customer_coords[i]
        lat2, lon2 = customer_coords[j]

        # Risiko nur wenn j ein Kunde ist und wir auf beladener Tour sind
        # Vereinfachung:
        # Depot -> Kunde und Kunde -> Kunde: loaded=True
        # Kunde -> Depot: loaded=False
        loaded_flag = (j != 0)

        relation_alts[i][j] = {}
        P_rel[i][j] = []

        profiles = list(PATH_PROFILES.keys()) if loaded_flag else ["shortest"]

        raw_alts = []
        for profile in profiles:
            alt = relation_metrics(lat1, lon1, lat2, lon2, profile=profile, loaded=loaded_flag)
            raw_alts.append({
                "path_id": profile,
                **alt
            })

        raw_alts = deduplicate_alternatives(raw_alts)

        for idx, alt in enumerate(raw_alts):
            p_name = f"{alt['path_id']}_{idx}"
            relation_alts[i][j][p_name] = alt
            P_rel[i][j].append(p_name)

logger.info("Relationen zwischen Depot und Kunden erzeugt.")

# ============================================================
# DIAGNOSE RELATIONEN
# ============================================================
print("\n" + "=" * 80)
print("CHECK RELATIONEN DEPOT -> KUNDE")
print("=" * 80)

for j in C:
    print(f"\nDepot -> Kunde {j}")
    for p in P_rel[0][j]:
        alt = relation_alts[0][j][p]
        print(
            f"  {p:18s} | Distanz = {alt['dist_km']:.2f} km | "
            f"Zeit = {alt['time_min']:.2f} min | "
            f"Risiko = {alt['risk']:.4f}"
        )

# ============================================================
# MODELL
# ============================================================
model = pl.LpProblem("Berlin_Hazmat_CVRP", pl.LpMinimize)

# x[i,j,v]
x = pl.LpVariable.dicts(
    "x",
    [(i, j, v) for i in N for j in N for v in V if i != j],
    cat="Binary"
)

# y[i,j,v,p]
y = pl.LpVariable.dicts(
    "y",
    [(i, j, v, p) for i in N for j in N for v in V if i != j for p in P_rel[i][j]],
    cat="Binary"
)

# ✅ ZEIT JE FAHRZEUG
T = pl.LpVariable.dicts(
    "T",
    [(i, v) for i in C for v in V],
    lowBound=0,
    cat="Continuous"
)

# ✅ MTZ JE FAHRZEUG
U = pl.LpVariable.dicts(
    "U",
    [(i, v) for i in C for v in V],
    lowBound=0,
    upBound=len(C),
    cat="Continuous"
)

# ============================================================
# ZIELFUNKTION
# ============================================================
def vehicle_cost_per_km(v):
    return VarCost[v] + EnergyPerKm[v] * EnergyPrice[v]

travel_cost = pl.lpSum(
    vehicle_cost_per_km(v) * relation_alts[i][j][p]["dist_km"] * y[(i, j, v, p)]
    for i in N for j in N for v in V if i != j for p in P_rel[i][j]
)

risk_cost = pl.lpSum(
    relation_alts[i][j][p]["risk"] * y[(i, j, v, p)]
    for i in N for j in N for v in V if i != j for p in P_rel[i][j]
)

fixed_cost = pl.lpSum(
    FixCost[v] * pl.lpSum(x[(0, j, v)] for j in C)
    for v in V
)

total_cost = travel_cost + fixed_cost
total_risk = risk_cost

cost_scale = max(1.0, len(C) * 100.0)
risk_scale = max(1.0, len(C) * 500.0)

model += (w_cost / cost_scale) * total_cost + (w_risk / risk_scale) * total_risk

# ============================================================
# NEBENBEDINGUNGEN
# ============================================================

# Jeder Kunde genau einmal bedienen
for j in C:
    model += pl.lpSum(x[(i, j, v)] for i in N if i != j for v in V) == 1

# Flusserhaltung
for v in V:
    for h in C:
        model += (
            pl.lpSum(x[(i, h, v)] for i in N if i != h) ==
            pl.lpSum(x[(h, j, v)] for j in N if j != h)
        )

# ✅ DEPOTFLOW FIX (entscheidend!)
for v in V:
    model += pl.lpSum(x[(0, j, v)] for j in C) == pl.lpSum(x[(i, 0, v)] for i in C)
    model += pl.lpSum(x[(0, j, v)] for j in C) <= 1

# Kopplung x und y
for i in N:
    for j in N:
        if i == j:
            continue
        for v in V:
            model += pl.lpSum(y[(i, j, v, p)] for p in P_rel[i][j]) == x[(i, j, v)]

# Kapazität
for v in V:
    model += pl.lpSum(
        Demand[j] * pl.lpSum(x[(i, j, v)] for i in N if i != j)
        for j in C
    ) <= Cap[v]

# ============================================================
# ZEIT
# ============================================================

BIG_M = 20000

# (optional stabiler machen)
for i in C:
    Latest[i] += 30   # kleine Relaxation

# ✅ ZEITFENSTER (aktiviert nur bei Besuch)
for i in C:
    for v in V:
        visit_expr = pl.lpSum(x[(k, i, v)] for k in N if k != i)

        model += T[(i, v)] >= Earliest[i] - BIG_M * (1 - visit_expr)
        model += T[(i, v)] <= Latest[i]   + BIG_M * (1 - visit_expr)

# ✅ START
for v in V:
    for j in C:
        travel_time = pl.lpSum(
            relation_alts[0][j][p]["time_min"] * y[(0, j, v, p)]
            for p in P_rel[0][j]
        )

        model += (
            T[(j, v)] >= ShiftStart[v] + travel_time
            - BIG_M * (1 - x[(0, j, v)])
        )

# ✅ KUNDE → KUNDE
for v in V:
    for i in C:
        for j in C:
            if i == j:
                continue

            travel_time = pl.lpSum(
                relation_alts[i][j][p]["time_min"] * y[(i, j, v, p)]
                for p in P_rel[i][j]
            )

            model += (
                T[(j, v)] >= T[(i, v)] + Service[i] + travel_time
                - BIG_M * (1 - x[(i, j, v)])
            )

# ✅ WICHTIG: Schichtende temporär entfernt (Feasibility!)
# später wieder aktivieren mit Puffer!

# ============================================================
# MTZ (vehikel-spezifisch)
# ============================================================

for v in V:
    for i in C:
        model += U[(i, v)] >= 1
        model += U[(i, v)] <= len(C)

for v in V:
    for i in C:
        for j in C:
            if i == j:
                continue

            model += (
                U[(j, v)] >= U[(i, v)] + 1
                - len(C) * (1 - x[(i, j, v)])
            )

# ============================================================
# SOLVER
# ============================================================
logger.info("Starte Optimierung...")

solver = pl.PULP_CBC_CMD(
    msg=1,
    timeLimit=300,
    gapRel=0.01
)

status = model.solve(solver)
status_str = pl.LpStatus[status]

print("\n" + "=" * 80)
print("SOLVER STATUS")
print("=" * 80)
print("Status:", status_str)
print("Objective:", pl.value(model.objective))
print(f"Gewichtung: Kosten = {w_cost:.2f}, Risiko = {w_risk:.2f}")

# ============================================================
# ROUTEN REKONSTRUIEREN
# ============================================================
def extract_vehicle_route(v):
    starts = [j for j in C if pl.value(x[(0, j, v)]) is not None and pl.value(x[(0, j, v)]) > 0.5]

    if not starts:
        return []

    route = [0]
    current = starts[0]
    route.append(current)

    visited = {current}

    while True:
        next_nodes = [
            j for j in N
            if j != current and pl.value(x[(current, j, v)]) is not None and pl.value(x[(current, j, v)]) > 0.5
        ]

        if not next_nodes:
            break

        nxt = next_nodes[0]
        route.append(nxt)

        if nxt == 0:
            break

        if nxt in visited:
            route.append("SUBTOUR")
            break

        visited.add(nxt)
        current = nxt

    return route

def selected_path(i, j, v):
    for p in P_rel[i][j]:
        val = pl.value(y[(i, j, v, p)])
        if val is not None and val > 0.5:
            return p
    return None

# ============================================================
# AUSWERTUNG
# ============================================================
vehicle_routes = {}

if status_str not in ["Optimal", "Feasible"]:
    print("\nKeine zulaessige Loesung gefunden.")
else:
    print("\n" + "=" * 80)
    print("FAHRZEUGSCHICHTEN")
    print("=" * 80)
    for v in V:
        print(f"{v}: {min_to_hhmm(ShiftStart[v])} - {min_to_hhmm(ShiftEnd[v])}")

    print("\n" + "=" * 80)
    print("ZEITFENSTER DER KUNDEN")
    print("=" * 80)
    for c in C:
        print(f"Kunde {c}: {min_to_hhmm(Earliest[c])} - {min_to_hhmm(Latest[c])}")

    print("\n" + "=" * 80)
    print("KUNDENSTARTZEITEN")
    print("=" * 80)

    for c in C:
        printed = False

        for v in V:
            val = pl.value(T[(c, v)])

            if val is not None and val > 0:
                print(f"Kunde {c} ({v}): {min_to_hhmm(val)}")
                printed = True

        if not printed:
            print(f"Kunde {c}: nicht bedient")

    print("\n" + "=" * 80)
    print("ROUTEN JE FAHRZEUG")
    print("=" * 80)

    for v in V:
        route = extract_vehicle_route(v)
        vehicle_routes[v] = route

        if route:
            print(f"{v}: " + " -> ".join(map(str, route)))
        else:
            print(f"{v}: nicht genutzt")

    print("\n" + "=" * 80)
    print("GEWAEHLTE RELATIONSPFADE")
    print("=" * 80)

    for v in V:
        route = vehicle_routes[v]
        if not route or len(route) < 2:
            continue

        for a, b in zip(route[:-1], route[1:]):
            if a == "SUBTOUR" or b == "SUBTOUR":
                continue
            p_sel = selected_path(a, b, v)
            if p_sel is None:
                continue
            alt = relation_alts[a][b][p_sel]
            print(
                f"{v}: {a} -> {b} | Pfad {p_sel} | "
                f"Distanz = {alt['dist_km']:.2f} km | "
                f"Zeit = {alt['time_min']:.2f} min | "
                f"Risiko = {alt['risk']:.4f}"
            )

    print("\n" + "=" * 80)
    print("KOSTEN / RISIKO")
    print("=" * 80)
    print(f"Gesamtkosten: {pl.value(total_cost):.2f}")
    print(f"Gesamtrisiko: {pl.value(total_risk):.4f}")

# ============================================================
# VISUALISIERUNG
# ============================================================
CENTER = (52.5200, 13.4050)

ROUTE_COLORS = {
    "Truck1": "#E63946",
    "Truck2": "#2196F3",
    "Truck3": "#4CAF50",
    "Truck4": "#FF9800",
    "Truck5": "#9C27B0"
}

def draw_edge_sequence(map_obj, edge_sequence, color, weight=5, opacity=0.9, tooltip_prefix=""):
    missing = 0

    for (u, v) in edge_sequence:
        coords = None

        if G_road.has_edge(u, v):
            coords = G_road[u][v].get("coords")

        if not coords:
            coord_u = node_coords.get(u)
            coord_v = node_coords.get(v)
            if coord_u and coord_v:
                coords = [coord_u, coord_v]

        if coords:
            folium.PolyLine(
                locations=coords,
                color=color,
                weight=weight,
                opacity=opacity,
                tooltip=f"{tooltip_prefix} | {u}->{v}"
            ).add_to(map_obj)
        else:
            missing += 1

    return missing

def build_map_for_vehicle(v, route):
    m = folium.Map(location=CENTER, zoom_start=12, tiles="CartoDB positron")
    MiniMap(toggle_display=True).add_to(m)

    pop_layer = folium.FeatureGroup(name="🏙️ Populationsdichte", show=False)
    HeatMap(
        pop_heat,
        min_opacity=0.2,
        radius=8,
        blur=6,
        max_zoom=15,
        gradient={0.0: "green", 0.5: "yellow", 1.0: "red"}
    ).add_to(pop_layer)
    pop_layer.add_to(m)

    acc_layer = folium.FeatureGroup(name="⚠️ Unfallrate", show=False)
    HeatMap(
        acc_heat,
        min_opacity=0.2,
        radius=8,
        blur=6,
        max_zoom=15,
        gradient={0.0: "green", 0.5: "yellow", 1.0: "red"}
    ).add_to(acc_layer)
    acc_layer.add_to(m)

    nat_layer = folium.FeatureGroup(name="🌿 Naturschutznähe", show=False)
    HeatMap(
        nat_heat,
        min_opacity=0.2,
        radius=8,
        blur=6,
        max_zoom=15,
        gradient={0.0: "green", 0.5: "yellow", 1.0: "red"}
    ).add_to(nat_layer)
    nat_layer.add_to(m)

    route_layer = folium.FeatureGroup(name=f"🚛 Route {v}", show=True)

    for a, b in zip(route[:-1], route[1:]):
        if a == "SUBTOUR" or b == "SUBTOUR":
            continue

        p_sel = selected_path(a, b, v)
        if p_sel is None:
            continue

        alt = relation_alts[a][b][p_sel]
        risk_val = alt["risk"]

        if b == 0:
            color = "#000000"
            weight = 4
        else:
            color = risk_to_color(risk_val, 0.0, 10.0)
            weight = 6

        draw_edge_sequence(
            route_layer,
            alt["edge_list"],
            color=color,
            weight=weight,
            opacity=0.9,
            tooltip_prefix=f"{a}->{b} | Risiko {risk_val:.4f}"
        )

    route_layer.add_to(m)

    # Depot Marker
    folium.Marker(
        location=(depot_lat, depot_lon),
        popup=folium.Popup(f"<b>Depot</b><br>Fahrzeug: {v}", max_width=220),
        icon=folium.Icon(color="blue", icon="home", prefix="fa")
    ).add_to(m)

    # Kunden Marker
    for c in C:
        coords = customer_coords[c]
        visited_by_v = any(
            pl.value(x[(i, c, v)]) is not None and pl.value(x[(i, c, v)]) > 0.5
            for i in N if i != c
        )

    if visited_by_v:
    # ✅ richtige Startzeit für dieses Fahrzeug holen
        start_time = pl.value(T[(c, v)])

        folium.Marker(
            location=coords,
            popup=folium.Popup(
                f"<b>Kunde {c}</b><br>"
                f"Fahrzeug: {v}<br>"
                f"Nachfrage: {Demand[c]} kg<br>"
                f"Startzeit: {min_to_hhmm(start_time) if start_time else 'n/a'}",
                max_width=260
            ),
            icon=folium.Icon(color="red", icon="truck", prefix="fa")
        ).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    return m

# ============================================================
# KARTEN SPEICHERN
# ============================================================
if status_str in ["Optimal", "Feasible"]:
    for v, route in vehicle_routes.items():
        if not route:
            continue

        print("\n" + "=" * 80)
        print(f"KARTE FUER {v}")
        print(f"Route: {' -> '.join(map(str, route))}")
        print("=" * 80)

        m = build_map_for_vehicle(v, route)

        filename = f"route_fahrzeug_{v}_cost_{w_cost:.2f}_risk_{w_risk:.2f}.html"
        m.save(filename)
        print(f"→ Gespeichert als: {filename}")


2026-06-27 19:28:37,343 - INFO - Daten erfolgreich geladen in 0.20s.
2026-06-27 19:28:39,127 - INFO - Knoten: 294,724
2026-06-27 19:28:39,127 - INFO - Kanten: 533,453
2026-06-27 19:30:51,977 - INFO - G_road aufgebaut: 294,724 Knoten, 533,453 Kanten
2026-06-27 19:46:46,224 - INFO - Relationen zwischen Depot und Kunden erzeugt.

CHECK RELATIONEN DEPOT -> KUNDE

Depot -> Kunde 1
  shortest_0         | Distanz = 3.18 km | Zeit = 5.45 min | Risiko = 2.2124
  balanced_1_1       | Distanz = 3.44 km | Zeit = 5.91 min | Risiko = 1.3119
  balanced_2_2       | Distanz = 3.50 km | Zeit = 6.00 min | Risiko = 1.2114
  safest_3           | Distanz = 4.48 km | Zeit = 7.68 min | Risiko = 0.9551

Depot -> Kunde 2
  shortest_0         | Distanz = 7.50 km | Zeit = 12.86 min | Risiko = 2.8021
  balanced_1_1       | Distanz = 7.82 km | Zeit = 13.41 min | Risiko = 1.5318
  balanced_2_2       | Distanz = 8.36 km | Zeit = 14.34 min | Risiko = 0.7589

Depot -> Kunde 3
  shortest_0         | Distanz = 6.39 km | 

CoPilot Paper Update

In [ ]:
# ============================================================
# BERLIN HAZMAT-CVRP PROTOTYP
# - 1 Depot
# - mehrere Kunden pro Route
# - 2 LKWs
# - Zeitfenster
# - Risiko + Kosten
# - Berliner Netzrelationen
# - echte Fahrzeugtouren
# ============================================================

import pandas as pd
import numpy as np
import pulp as pl
import networkx as nx
import logging
import sys
import pickle
import time
import folium

from folium.plugins import MiniMap, HeatMap
from math import radians, sin, cos, sqrt, atan2

# ============================================================
# LOGGING
# ============================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
    force=True
)
logger = logging.getLogger(__name__)

# ============================================================
# ZEITFUNKTIONEN
# ============================================================
def hhmm_to_min(hhmm):
    h, m = map(int, hhmm.split(":"))
    return 60 * h + m

def min_to_hhmm(minutes):
    minutes = int(round(minutes))
    h = minutes // 60
    m = minutes % 60
    return f"{h:02d}:{m:02d}"

# ============================================================
# GEWICHTUNG
# ============================================================
w_cost = 0.50
w_risk = 0.50

# Beispiele:
# w_cost = 0.90 ; w_risk = 0.10
# w_cost = 0.50 ; w_risk = 0.50
# w_cost = 0.10 ; w_risk = 0.90

# ============================================================
# DATEN LADEN
# ============================================================
t0 = time.perf_counter()

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_with_accidents_new.pkl", "rb") as f:
    berlin_graph_accidents = pickle.load(f)

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_with_nature_new.pkl", "rb") as f:
    berlin_graph_nature = pickle.load(f)

with open(r"C:\Users\tspol\Sciebo\OperationsResearch\berlin_graph_with_population_new.pkl", "rb") as f:
    berlin_graph_population = pickle.load(f)

logger.info(f"Daten erfolgreich geladen in {time.perf_counter() - t0:.2f}s.")

# ============================================================
# HILFSFUNKTIONEN
# ============================================================
def haversine_scalar(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def normalize(values_dict, invert=False):
    vals = np.array(list(values_dict.values()), dtype=float)
    if len(vals) == 0:
        return {}
    vmin = vals.min()
    vmax = vals.max()

    if vmax == vmin:
        return {k: 0.0 for k in values_dict}

    out = {}
    for k, v in values_dict.items():
        x = (v - vmin) / (vmax - vmin + 1e-9)
        if invert:
            x = 1.0 - x
        out[k] = float(x)
    return out

def risk_to_color(value, vmin=0.0, vmax=1.0):
    if vmax == vmin:
        x = 0.0
    else:
        x = (value - vmin) / (vmax - vmin)
    x = max(0.0, min(1.0, x))
    r = int(255 * x)
    g = int(255 * (1 - x))
    return f"#{r:02x}{g:02x}00"

# ============================================================
# NETZ AUFBAUEN
# ============================================================
df_nodes = berlin_graph_accidents["nodes"].copy()
df_edges = berlin_graph_accidents["edges"].copy()

node_coords = (
    df_nodes
    .set_index("node")[["lat", "lon"]]
    .apply(tuple, axis=1)
    .to_dict()
)

node_ids = df_nodes["node"].values
node_lats = df_nodes["lat"].values
node_lons = df_nodes["lon"].values

def nearest_node(lat, lon):
    dists = haversine_vectorized(lat, lon, node_lats, node_lons)
    idx = np.argmin(dists)
    return int(node_ids[idx]), float(dists[idx])

logger.info(f"Knoten: {len(df_nodes):,}")
logger.info(f"Kanten: {len(df_edges):,}")

# ============================================================
# RISIKODATEN
# ============================================================
pop_dict = dict(zip(
    zip(berlin_graph_population["edges"]["u"], berlin_graph_population["edges"]["v"]),
    berlin_graph_population["edges"]["pop_per_meter"]
))

acc_dict = dict(zip(
    zip(berlin_graph_accidents["edges"]["u"], berlin_graph_accidents["edges"]["v"]),
    berlin_graph_accidents["edges"]["score"]
))

nat_dict = dict(zip(
    zip(berlin_graph_nature["edges"]["u"], berlin_graph_nature["edges"]["v"]),
    berlin_graph_nature["edges"]["dist_to_nature_m"]
))

pop_norm = normalize(pop_dict)
acc_norm = normalize(acc_dict)
nat_norm = normalize(nat_dict, invert=True)

alpha = 0.4
beta  = 0.4
gamma = 0.2

edge_risk_base = {}

for _, row in df_edges.iterrows():
    e = (int(row["u"]), int(row["v"]))
    pop = pop_norm.get(e, 0.0)
    acc = acc_norm.get(e, 0.0)
    nat = nat_norm.get(e, 0.0)

    consequence = alpha * pop + gamma * nat
    probability = beta * acc
    edge_risk_base[e] = probability * consequence

# ============================================================
# STRASSENGRAPH
# ============================================================
G_road = nx.DiGraph()

for _, row in df_edges.iterrows():
    u = int(row["u"])
    v = int(row["v"])

    if "distance" in df_edges.columns:
        dist_km = float(row["distance"]) / 1000.0
    elif "length" in df_edges.columns:
        dist_km = float(row["length"]) / 1000.0
    else:
        coord_u = node_coords.get(u)
        coord_v = node_coords.get(v)
        if coord_u is None or coord_v is None:
            continue
        dist_km = haversine_scalar(coord_u[0], coord_u[1], coord_v[0], coord_v[1]) / 1000.0

    coords = None
    if u in node_coords and v in node_coords:
        coords = [node_coords[u], node_coords[v]]

    G_road.add_edge(
        u, v,
        distance_km=dist_km,
        accident_score=acc_norm.get((u, v), 0.0),
        population_score=pop_norm.get((u, v), 0.0),
        nature_score=nat_norm.get((u, v), 0.0),
        base_risk=edge_risk_base.get((u, v), 0.0),
        coords=coords
    )

logger.info(f"G_road aufgebaut: {G_road.number_of_nodes():,} Knoten, {G_road.number_of_edges():,} Kanten")

# ============================================================
# HEATMAP-DATEN
# ============================================================
def build_heatmap_data(norm_dict):
    heat_data = []
    for (u, v), norm_val in norm_dict.items():
        coord_u = node_coords.get(u)
        coord_v = node_coords.get(v)
        if coord_u and coord_v and norm_val > 0:
            mid_lat = (coord_u[0] + coord_v[0]) / 2
            mid_lon = (coord_u[1] + coord_v[1]) / 2
            heat_data.append([mid_lat, mid_lon, float(norm_val)])
    return heat_data

pop_heat = build_heatmap_data(pop_norm)
acc_heat = build_heatmap_data(acc_norm)
nat_heat = build_heatmap_data(nat_norm)

# ============================================================
# PFADPROFILE
# ============================================================
PATH_PROFILES = {
    "shortest":   0.0,
    "balanced_1": 0.5,
    "balanced_2": 1.0,
    "balanced_3": 2.0,
    "safest":     4.0
}

AVG_SPEED_KMH = 35.0
path_cache = {}

def relation_metrics(lat1, lon1, lat2, lon2, profile="shortest", loaded=True):
    """
    loaded=True  -> Risiko wirkt
    loaded=False -> Rueckfahrt/leer, kein Risiko
    """
    start_node, start_offset_m = nearest_node(lat1, lon1)
    end_node, end_offset_m = nearest_node(lat2, lon2)

    cache_key = (start_node, end_node, profile, loaded)
    if cache_key in path_cache:
        return path_cache[cache_key]

    H = nx.DiGraph()
    lam = PATH_PROFILES[profile]

    for u, v, data in G_road.edges(data=True):
        dist_km = data["distance_km"]
        risk_val = data["base_risk"] * 100.0 if loaded else 0.0

        H.add_edge(
            u, v,
            distance_km=dist_km,
            risk_val=risk_val,
            path_weight=dist_km + lam * risk_val
        )

    try:
        path_nodes = nx.shortest_path(H, source=start_node, target=end_node, weight="path_weight")
    except nx.NetworkXNoPath:
        result = {
            "dist_km": 1e6,
            "time_min": 1e6,
            "risk": 1e6,
            "edge_list": [],
            "start_node": start_node,
            "end_node": end_node
        }
        path_cache[cache_key] = result
        return result

    edge_list = list(zip(path_nodes[:-1], path_nodes[1:]))

    dist_km = 0.0
    risk_sum = 0.0

    for e in edge_list:
        ed = H[e[0]][e[1]]
        dist_km += ed["distance_km"]
        risk_sum += ed["risk_val"]

    dist_km += (start_offset_m + end_offset_m) / 1000.0
    time_min = 60.0 * dist_km / AVG_SPEED_KMH

    result = {
        "dist_km": dist_km,
        "time_min": time_min,
        "risk": risk_sum,
        "edge_list": edge_list,
        "start_node": start_node,
        "end_node": end_node
    }
    path_cache[cache_key] = result
    return result

# ============================================================
# DEPOT / KUNDEN / FAHRZEUGE
# ============================================================
depot_lat = 52.5200
depot_lon = 13.4050

customers = pd.DataFrame([
    {
        "customer_id": 1,
        "lat": 52.4986,
        "lon": 13.4030,
        "demand_kg": 6000,
        "service_time_min": 45,
        "earliest_min": hhmm_to_min("08:30"),
        "latest_min": hhmm_to_min("12:00")
    },
    {
        "customer_id": 2,
        "lat": 52.5163,
        "lon": 13.3041,
        "demand_kg": 5000,
        "service_time_min": 45,
        "earliest_min": hhmm_to_min("09:00"),
        "latest_min": hhmm_to_min("12:30")
    },
    {
        "customer_id": 3,
        "lat": 52.5695,
        "lon": 13.4010,
        "demand_kg": 4000,
        "service_time_min": 45,
        "earliest_min": hhmm_to_min("09:30"),
        "latest_min": hhmm_to_min("13:00")
    },
    {
        "customer_id": 4,
        "lat": 52.4800,
        "lon": 13.4500,
        "demand_kg": 3000,
        "service_time_min": 45,
        "earliest_min": hhmm_to_min("08:45"),
        "latest_min": hhmm_to_min("13:15")
    },
    {
        "customer_id": 5,
        "lat": 52.4700,
        "lon": 13.3300,
        "demand_kg": 4500,
        "service_time_min": 45,
        "earliest_min": hhmm_to_min("10:00"),
        "latest_min": hhmm_to_min("14:00")
    },
    {
        "customer_id": 6,
        "lat": 52.5600,
        "lon": 13.2800,
        "demand_kg": 3500,
        "service_time_min": 45,
        "earliest_min": hhmm_to_min("10:15"),
        "latest_min": hhmm_to_min("14:30")
    }
])

vehicles = pd.DataFrame([
    {
        "vehicle_id": "Truck1",
        "capacity_kg": 16000,
        "fixed_cost": 220,
        "variable_cost_per_km": 1.20,
        "energy_kwh_per_km": 1.50,
        "energy_price": 0.35,
        "shift_start_min": hhmm_to_min("08:00"),
        "shift_end_min": hhmm_to_min("15:00")
    },
    {
        "vehicle_id": "Truck2",
        "capacity_kg": 12000,
        "fixed_cost": 180,
        "variable_cost_per_km": 1.00,
        "energy_kwh_per_km": 1.20,
        "energy_price": 0.35,
        "shift_start_min": hhmm_to_min("08:00"),
        "shift_end_min": hhmm_to_min("15:00")
    }
])

C = customers["customer_id"].tolist()
V = vehicles["vehicle_id"].tolist()
N = [0] + C  # 0 = Depot

Demand = dict(zip(customers["customer_id"], customers["demand_kg"]))
Service = dict(zip(customers["customer_id"], customers["service_time_min"]))
Earliest = dict(zip(customers["customer_id"], customers["earliest_min"]))
Latest = dict(zip(customers["customer_id"], customers["latest_min"]))

Cap = dict(zip(vehicles["vehicle_id"], vehicles["capacity_kg"]))
FixCost = dict(zip(vehicles["vehicle_id"], vehicles["fixed_cost"]))
ShiftStart = dict(zip(vehicles["vehicle_id"], vehicles["shift_start_min"]))
ShiftEnd = dict(zip(vehicles["vehicle_id"], vehicles["shift_end_min"]))
VarCost = dict(zip(vehicles["vehicle_id"], vehicles["variable_cost_per_km"]))
EnergyPerKm = dict(zip(vehicles["vehicle_id"], vehicles["energy_kwh_per_km"]))
EnergyPrice = dict(zip(vehicles["vehicle_id"], vehicles["energy_price"]))

customer_coords = {
    row["customer_id"]: (row["lat"], row["lon"])
    for _, row in customers.iterrows()
}
customer_coords[0] = (depot_lat, depot_lon)

# ============================================================
# RELATIONEN ZWISCHEN DEPOT/KUNDEN
# ============================================================
relation_alts = {}
P_rel = {}

for i in N:
    relation_alts[i] = {}
    P_rel[i] = {}

    for j in N:
        if i == j:
            continue

        lat1, lon1 = customer_coords[i]
        lat2, lon2 = customer_coords[j]

        # Risiko nur wenn j ein Kunde ist und wir auf beladener Tour sind
        # Vereinfachung:
        # Depot -> Kunde und Kunde -> Kunde: loaded=True
        # Kunde -> Depot: loaded=False
        loaded_flag = (j != 0)

        relation_alts[i][j] = {}
        P_rel[i][j] = []

        profiles = list(PATH_PROFILES.keys()) if loaded_flag else ["shortest"]

        raw_alts = []
        for profile in profiles:
            alt = relation_metrics(lat1, lon1, lat2, lon2, profile=profile, loaded=loaded_flag)
            raw_alts.append({
                "path_id": profile,
                **alt
            })

        raw_alts = deduplicate_alternatives(raw_alts)

        for idx, alt in enumerate(raw_alts):
            p_name = f"{alt['path_id']}_{idx}"
            relation_alts[i][j][p_name] = alt
            P_rel[i][j].append(p_name)

logger.info("Relationen zwischen Depot und Kunden erzeugt.")

# ============================================================
# DIAGNOSE RELATIONEN
# ============================================================
print("\n" + "=" * 80)
print("CHECK RELATIONEN DEPOT -> KUNDE")
print("=" * 80)

for j in C:
    print(f"\nDepot -> Kunde {j}")
    for p in P_rel[0][j]:
        alt = relation_alts[0][j][p]
        print(
            f"  {p:18s} | Distanz = {alt['dist_km']:.2f} km | "
            f"Zeit = {alt['time_min']:.2f} min | "
            f"Risiko = {alt['risk']:.4f}"
        )

# ============================================================
# MODELL
# ============================================================
model = pl.LpProblem("Berlin_Hazmat_CVRP", pl.LpMinimize)

# x[i,j,v]
x = pl.LpVariable.dicts(
    "x",
    [(i, j, v) for i in N for j in N for v in V if i != j],
    cat="Binary"
)

# y[i,j,v,p]
y = pl.LpVariable.dicts(
    "y",
    [(i, j, v, p) for i in N for j in N for v in V if i != j for p in P_rel[i][j]],
    cat="Binary"
)

# ✅ ZEIT JE FAHRZEUG
T = pl.LpVariable.dicts(
    "T",
    [(i, v) for i in C for v in V],
    lowBound=0,
    cat="Continuous"
)

# ✅ MTZ JE FAHRZEUG
U = pl.LpVariable.dicts(
    "U",
    [(i, v) for i in C for v in V],
    lowBound=0,
    upBound=len(C),
    cat="Continuous"
)

# ============================================================
# ZIELFUNKTION
# ============================================================
def vehicle_cost_per_km(v):
    return VarCost[v] + EnergyPerKm[v] * EnergyPrice[v]

travel_cost = pl.lpSum(
    vehicle_cost_per_km(v) * relation_alts[i][j][p]["dist_km"] * y[(i, j, v, p)]
    for i in N for j in N for v in V if i != j for p in P_rel[i][j]
)

risk_cost = pl.lpSum(
    relation_alts[i][j][p]["risk"] * y[(i, j, v, p)]
    for i in N for j in N for v in V if i != j for p in P_rel[i][j]
)

fixed_cost = pl.lpSum(
    FixCost[v] * pl.lpSum(x[(0, j, v)] for j in C)
    for v in V
)

total_cost = travel_cost + fixed_cost
total_risk = risk_cost

cost_scale = max(1.0, len(C) * 100.0)
risk_scale = max(1.0, len(C) * 500.0)

model += (w_cost / cost_scale) * total_cost + (w_risk / risk_scale) * total_risk

# ============================================================
# NEBENBEDINGUNGEN
# ============================================================

# Jeder Kunde genau einmal bedienen
for j in C:
    model += pl.lpSum(x[(i, j, v)] for i in N if i != j for v in V) == 1

# Flusserhaltung
for v in V:
    for h in C:
        model += (
            pl.lpSum(x[(i, h, v)] for i in N if i != h) ==
            pl.lpSum(x[(h, j, v)] for j in N if j != h)
        )

# ✅ DEPOTFLOW FIX (entscheidend!)
for v in V:
    model += pl.lpSum(x[(0, j, v)] for j in C) == pl.lpSum(x[(i, 0, v)] for i in C)
    model += pl.lpSum(x[(0, j, v)] for j in C) <= 1

# Kopplung x und y
for i in N:
    for j in N:
        if i == j:
            continue
        for v in V:
            model += pl.lpSum(y[(i, j, v, p)] for p in P_rel[i][j]) == x[(i, j, v)]

# Kapazität
for v in V:
    model += pl.lpSum(
        Demand[j] * pl.lpSum(x[(i, j, v)] for i in N if i != j)
        for j in C
    ) <= Cap[v]

# ============================================================
# ZEIT
# ============================================================

BIG_M = 20000

# (optional stabiler machen)
for i in C:
    Latest[i] += 30   # kleine Relaxation

# ✅ ZEITFENSTER (aktiviert nur bei Besuch)
for i in C:
    for v in V:
        visit_expr = pl.lpSum(x[(k, i, v)] for k in N if k != i)

        model += T[(i, v)] >= Earliest[i] - BIG_M * (1 - visit_expr)
        model += T[(i, v)] <= Latest[i]   + BIG_M * (1 - visit_expr)

# ✅ START
for v in V:
    for j in C:
        travel_time = pl.lpSum(
            relation_alts[0][j][p]["time_min"] * y[(0, j, v, p)]
            for p in P_rel[0][j]
        )

        model += (
            T[(j, v)] >= ShiftStart[v] + travel_time
            - BIG_M * (1 - x[(0, j, v)])
        )

# ✅ KUNDE → KUNDE
for v in V:
    for i in C:
        for j in C:
            if i == j:
                continue

            travel_time = pl.lpSum(
                relation_alts[i][j][p]["time_min"] * y[(i, j, v, p)]
                for p in P_rel[i][j]
            )

            model += (
                T[(j, v)] >= T[(i, v)] + Service[i] + travel_time
                - BIG_M * (1 - x[(i, j, v)])
            )

# ✅ WICHTIG: Schichtende temporär entfernt (Feasibility!)
# später wieder aktivieren mit Puffer!

# ============================================================
# MTZ (vehikel-spezifisch)
# ============================================================

for v in V:
    for i in C:
        model += U[(i, v)] >= 1
        model += U[(i, v)] <= len(C)

for v in V:
    for i in C:
        for j in C:
            if i == j:
                continue

            model += (
                U[(j, v)] >= U[(i, v)] + 1
                - len(C) * (1 - x[(i, j, v)])
            )

# ============================================================
# SOLVER
# ============================================================
logger.info("Starte Optimierung...")

solver = pl.PULP_CBC_CMD(
    msg=1,
    timeLimit=300,
    gapRel=0.01
)

status = model.solve(solver)
status_str = pl.LpStatus[status]

print("\n" + "=" * 80)
print("SOLVER STATUS")
print("=" * 80)
print("Status:", status_str)
print("Objective:", pl.value(model.objective))
print(f"Gewichtung: Kosten = {w_cost:.2f}, Risiko = {w_risk:.2f}")

# ============================================================
# ROUTEN REKONSTRUIEREN
# ============================================================
def extract_vehicle_route(v):
    starts = [j for j in C if pl.value(x[(0, j, v)]) is not None and pl.value(x[(0, j, v)]) > 0.5]

    if not starts:
        return []

    route = [0]
    current = starts[0]
    route.append(current)

    visited = {current}

    while True:
        next_nodes = [
            j for j in N
            if j != current and pl.value(x[(current, j, v)]) is not None and pl.value(x[(current, j, v)]) > 0.5
        ]

        if not next_nodes:
            break

        nxt = next_nodes[0]
        route.append(nxt)

        if nxt == 0:
            break

        if nxt in visited:
            route.append("SUBTOUR")
            break

        visited.add(nxt)
        current = nxt

    return route

def selected_path(i, j, v):
    for p in P_rel[i][j]:
        val = pl.value(y[(i, j, v, p)])
        if val is not None and val > 0.5:
            return p
    return None

# ============================================================
# AUSWERTUNG
# ============================================================
vehicle_routes = {}

if status_str not in ["Optimal", "Feasible"]:
    print("\nKeine zulaessige Loesung gefunden.")
else:
    print("\n" + "=" * 80)
    print("FAHRZEUGSCHICHTEN")
    print("=" * 80)
    for v in V:
        print(f"{v}: {min_to_hhmm(ShiftStart[v])} - {min_to_hhmm(ShiftEnd[v])}")

    print("\n" + "=" * 80)
    print("ZEITFENSTER DER KUNDEN")
    print("=" * 80)
    for c in C:
        print(f"Kunde {c}: {min_to_hhmm(Earliest[c])} - {min_to_hhmm(Latest[c])}")

    print("\n" + "=" * 80)
    print("KUNDENSTARTZEITEN")
    print("=" * 80)

    for c in C:
        printed = False

        for v in V:
            val = pl.value(T[(c, v)])

            if val is not None and val > 0:
                print(f"Kunde {c} ({v}): {min_to_hhmm(val)}")
                printed = True

        if not printed:
            print(f"Kunde {c}: nicht bedient")

    print("\n" + "=" * 80)
    print("ROUTEN JE FAHRZEUG")
    print("=" * 80)

    for v in V:
        route = extract_vehicle_route(v)
        vehicle_routes[v] = route

        if route:
            print(f"{v}: " + " -> ".join(map(str, route)))
        else:
            print(f"{v}: nicht genutzt")

    print("\n" + "=" * 80)
    print("GEWAEHLTE RELATIONSPFADE")
    print("=" * 80)

    for v in V:
        route = vehicle_routes[v]
        if not route or len(route) < 2:
            continue

        for a, b in zip(route[:-1], route[1:]):
            if a == "SUBTOUR" or b == "SUBTOUR":
                continue
            p_sel = selected_path(a, b, v)
            if p_sel is None:
                continue
            alt = relation_alts[a][b][p_sel]
            print(
                f"{v}: {a} -> {b} | Pfad {p_sel} | "
                f"Distanz = {alt['dist_km']:.2f} km | "
                f"Zeit = {alt['time_min']:.2f} min | "
                f"Risiko = {alt['risk']:.4f}"
            )

    print("\n" + "=" * 80)
    print("KOSTEN / RISIKO")
    print("=" * 80)
    print(f"Gesamtkosten: {pl.value(total_cost):.2f}")
    print(f"Gesamtrisiko: {pl.value(total_risk):.4f}")

# ============================================================
# VISUALISIERUNG
# ============================================================
CENTER = (52.5200, 13.4050)

ROUTE_COLORS = {
    "Truck1": "#E63946",
    "Truck2": "#2196F3",
    "Truck3": "#4CAF50",
    "Truck4": "#FF9800",
    "Truck5": "#9C27B0"
}

def draw_edge_sequence(map_obj, edge_sequence, color, weight=5, opacity=0.9, tooltip_prefix=""):
    missing = 0

    for (u, v) in edge_sequence:
        coords = None

        if G_road.has_edge(u, v):
            coords = G_road[u][v].get("coords")

        if not coords:
            coord_u = node_coords.get(u)
            coord_v = node_coords.get(v)
            if coord_u and coord_v:
                coords = [coord_u, coord_v]

        if coords:
            folium.PolyLine(
                locations=coords,
                color=color,
                weight=weight,
                opacity=opacity,
                tooltip=f"{tooltip_prefix} | {u}->{v}"
            ).add_to(map_obj)
        else:
            missing += 1

    return missing

def build_map_for_vehicle(v, route):
    m = folium.Map(location=CENTER, zoom_start=12, tiles="CartoDB positron")
    MiniMap(toggle_display=True).add_to(m)

    pop_layer = folium.FeatureGroup(name="🏙️ Populationsdichte", show=False)
    HeatMap(
        pop_heat,
        min_opacity=0.2,
        radius=8,
        blur=6,
        max_zoom=15,
        gradient={0.0: "green", 0.5: "yellow", 1.0: "red"}
    ).add_to(pop_layer)
    pop_layer.add_to(m)

    acc_layer = folium.FeatureGroup(name="⚠️ Unfallrate", show=False)
    HeatMap(
        acc_heat,
        min_opacity=0.2,
        radius=8,
        blur=6,
        max_zoom=15,
        gradient={0.0: "green", 0.5: "yellow", 1.0: "red"}
    ).add_to(acc_layer)
    acc_layer.add_to(m)

    nat_layer = folium.FeatureGroup(name="🌿 Naturschutznähe", show=False)
    HeatMap(
        nat_heat,
        min_opacity=0.2,
        radius=8,
        blur=6,
        max_zoom=15,
        gradient={0.0: "green", 0.5: "yellow", 1.0: "red"}
    ).add_to(nat_layer)
    nat_layer.add_to(m)

    route_layer = folium.FeatureGroup(name=f"🚛 Route {v}", show=True)

    for a, b in zip(route[:-1], route[1:]):
        if a == "SUBTOUR" or b == "SUBTOUR":
            continue

        p_sel = selected_path(a, b, v)
        if p_sel is None:
            continue

        alt = relation_alts[a][b][p_sel]
        risk_val = alt["risk"]

        if b == 0:
            color = "#000000"
            weight = 4
        else:
            color = risk_to_color(risk_val, 0.0, 10.0)
            weight = 6

        draw_edge_sequence(
            route_layer,
            alt["edge_list"],
            color=color,
            weight=weight,
            opacity=0.9,
            tooltip_prefix=f"{a}->{b} | Risiko {risk_val:.4f}"
        )

    route_layer.add_to(m)

    # Depot Marker
    folium.Marker(
        location=(depot_lat, depot_lon),
        popup=folium.Popup(f"<b>Depot</b><br>Fahrzeug: {v}", max_width=220),
        icon=folium.Icon(color="blue", icon="home", prefix="fa")
    ).add_to(m)

    # Kunden Marker
    for c in C:
        coords = customer_coords[c]
        visited_by_v = any(
            pl.value(x[(i, c, v)]) is not None and pl.value(x[(i, c, v)]) > 0.5
            for i in N if i != c
        )

    if visited_by_v:
    # ✅ richtige Startzeit für dieses Fahrzeug holen
        start_time = pl.value(T[(c, v)])

        folium.Marker(
            location=coords,
            popup=folium.Popup(
                f"<b>Kunde {c}</b><br>"
                f"Fahrzeug: {v}<br>"
                f"Nachfrage: {Demand[c]} kg<br>"
                f"Startzeit: {min_to_hhmm(start_time) if start_time else 'n/a'}",
                max_width=260
            ),
            icon=folium.Icon(color="red", icon="truck", prefix="fa")
        ).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    return m

# ============================================================
# KARTEN SPEICHERN
# ============================================================
if status_str in ["Optimal", "Feasible"]:
    for v, route in vehicle_routes.items():
        if not route:
            continue

        print("\n" + "=" * 80)
        print(f"KARTE FUER {v}")
        print(f"Route: {' -> '.join(map(str, route))}")
        print("=" * 80)

        m = build_map_for_vehicle(v, route)

        filename = f"route_fahrzeug_{v}_cost_{w_cost:.2f}_risk_{w_risk:.2f}.html"
        m.save(filename)
        print(f"→ Gespeichert als: {filename}")


In [7]:
# ============================================================
# FINAL VRP (MULTI-LKW + ZEIT + RISIKO + FIXKOSTEN)
# ============================================================

import pandas as pd
import pulp as pl

# ============================================================
# 1. OD MATRIX
# ============================================================

od_df = pd.read_csv(r"C:\Users\tspol\Sciebo\OperationsResearch\germany_od_matrix_test.csv")
od_df = od_df[od_df["profile"] == "safest"]

# ============================================================
# 2. DATENSTRUKTUREN
# ============================================================

dist, risk, time = {}, {}, {}
nodes = set()

for _, row in od_df.iterrows():
    i, j = row["from"], row["to"]

    dist[(i, j)] = row["dist_km"]
    risk[(i, j)] = row["risk"]
    time[(i, j)] = row["time_min"]

    nodes.add(i)
    nodes.add(j)

valid_arcs = list(dist.keys())

# ============================================================
# COMPLETE GRAPH (WICHTIG)
# ============================================================

nodes_list = list(nodes)

max_dist = max(dist.values())
max_risk = max(risk.values())
max_time = max(time.values())

for i in nodes_list:
    for j in nodes_list:
        if i != j and (i, j) not in dist:
            dist[(i, j)] = max_dist * 10
            risk[(i, j)] = max_risk * 10
            time[(i, j)] = max_time * 10
            valid_arcs.append((i, j))

# ============================================================
# SETS
# ============================================================

DEPOT = "DEPOT"
customers = [n for n in nodes if n != DEPOT]

vehicles = ["Truck1", "Truck2", "Truck3"]

# ============================================================
# PARAMETER
# ============================================================

Demand = {c: 2000 for c in customers}

Cap = {
    "Truck1": 10000,
    "Truck2": 12000,
    "Truck3": 8000
}

# 🔥 NEU: Fixkosten pro Fahrzeug
FixCost = {
    "Truck1": 300,
    "Truck2": 300,
    "Truck3": 300
}

ServiceTime = {c: 5 for c in customers}
MAX_TIME = 60000   # z.B. 10 Stunden

# ============================================================
# MODELL
# ============================================================

model = pl.LpProblem("Hazmat_VRP_MultiVehicle", pl.LpMinimize)

x = pl.LpVariable.dicts(
    "x",
    [(i, j, v) for (i, j) in valid_arcs for v in vehicles],
    cat="Binary"
)

T = pl.LpVariable.dicts(
    "T",
    [(i, v) for i in customers for v in vehicles],
    lowBound=0
)

# ============================================================
# ZIELFUNKTION
# ============================================================

w_cost = 0.3
w_risk = 0.7

travel_cost = pl.lpSum(
    (w_cost * dist[(i, j)] + w_risk * risk[(i, j)]) * x[(i, j, v)]
    for (i, j, v) in x
)

# ✅ Fixkosten nur wenn Fahrzeug benutzt wird
fixed_cost = pl.lpSum(
    FixCost[v] * pl.lpSum(x[(DEPOT, j, v)] for j in customers)
    for v in vehicles
)

model += travel_cost + fixed_cost

# ============================================================
# CONSTRAINTS
# ============================================================

# Jeder Kunde genau einmal
for j in customers:
    model += pl.lpSum(
        x[(i, j, v)]
        for i in nodes for v in vehicles if i != j
    ) == 1

# Flusserhaltung
for v in vehicles:
    for h in customers:
        model += (
            pl.lpSum(x[(i, h, v)] for i in nodes if i != h) ==
            pl.lpSum(x[(h, j, v)] for j in nodes if j != h)
        )

# Depotbindung
for v in vehicles:
    model += (
        pl.lpSum(x[(DEPOT, j, v)] for j in customers) ==
        pl.lpSum(x[(i, DEPOT, v)] for i in customers)
    )

    model += pl.lpSum(x[(DEPOT, j, v)] for j in customers) <= 1

# Kapazität
for v in vehicles:
    model += pl.lpSum(
        Demand[j] * pl.lpSum(x[(i, j, v)] for i in nodes if i != j)
        for j in customers
    ) <= Cap[v]

# ============================================================
# ZEITCONSTRAINTS
# ============================================================

BIG_M = 100000

# Start vom Depot
for v in vehicles:
    for j in customers:
        model += (
            T[(j, v)] >= time[(DEPOT, j)]
            - BIG_M * (1 - x[(DEPOT, j, v)])
        )

# Kunde → Kunde
for v in vehicles:
    for i in customers:
        for j in customers:
            if i != j:
                model += (
                    T[(j, v)] >=
                    T[(i, v)] + ServiceTime[i] + time[(i, j)]
                    - BIG_M * (1 - x[(i, j, v)])
                )

# Rückkehr zum Depot (WICHTIG!)
for v in vehicles:
    for i in customers:
        model += (
            T[(i, v)] + ServiceTime[i] + time[(i, DEPOT)]
            <= MAX_TIME + BIG_M * (1 - x[(i, DEPOT, v)])
        )

# maximale Tourdauer
for v in vehicles:
    model += pl.lpSum(
        time[(i, j)] * x[(i, j, v)]
        for (i, j, v2) in x if v2 == v
    ) <= MAX_TIME

# ============================================================
# MTZ SUBTOUR
# ============================================================

U = pl.LpVariable.dicts("U", customers, 0, len(customers))

for i in customers:
    for j in customers:
        if i != j:
            model += (
                U[i] - U[j]
                + len(customers) * pl.lpSum(x[(i, j, v)] for v in vehicles)
                <= len(customers) - 1
            )

# ============================================================
# SOLVER
# ============================================================

print("Starte Solver...")

model.solve(pl.PULP_CBC_CMD(msg=1, timeLimit=300))

print("\nStatus:", pl.LpStatus[model.status])
print("Objective:", pl.value(model.objective))

# ============================================================
# ROUTEN
# ============================================================

def build_route(v):
    route = [DEPOT]
    current = DEPOT
    visited = set()

    while True:
        next_nodes = [
            j for j in customers + [DEPOT]
            if j != current and pl.value(x[(current, j, v)]) > 0.5
        ]

        if not next_nodes:
            break

        nxt = next_nodes[0]
        route.append(nxt)

        if nxt == DEPOT:
            break

        if nxt in visited:
            route.append("SUBTOUR")
            break

        visited.add(nxt)
        current = nxt

    return route

print("\nROUTEN:")

for v in vehicles:
    r = build_route(v)

    if len(r) > 1:
        print(f"{v}: {' -> '.join(r)}")
    else:
        print(f"{v}: nicht genutzt")

Starte Solver...

Status: Optimal
Objective: 5522.021855248471

ROUTEN:
Truck1: nicht genutzt
Truck2: nicht genutzt
Truck3: DEPOT -> C1 -> C2 -> C4 -> C3 -> DEPOT


In [8]:
# ============================================================
# KOSTEN UND RISIKO AUSRECHNEN
# ============================================================

total_distance = 0
total_risk = 0

for (i, j, v) in x:
    val = pl.value(x[(i, j, v)])

    if val is not None and val > 0.5:
        total_distance += dist[(i, j)]
        total_risk += risk[(i, j)]

# gewichtete Kosten (wie im Modell)
total_cost = w_cost * total_distance + w_risk * total_risk

print("\n" + "="*60)
print("ERGEBNISSE")
print("="*60)

print(f"Gesamtdistanz: {total_distance:.2f} km")
print(f"Gesamtrisiko:  {total_risk:.2f}")
print(f"Kostenwert:    {total_cost:.2f}")


ERGEBNISSE
Gesamtdistanz: 14089.90 km
Gesamtrisiko:  1421.50
Kostenwert:    5222.02


In [4]:
# ============================================================
# KOSTEN UND RISIKO AUSRECHNEN
# ============================================================

total_distance = 0
total_risk = 0

for (i, j, v) in x:
    val = pl.value(x[(i, j, v)])

    if val is not None and val > 0.5:
        total_distance += dist[(i, j)]
        total_risk += risk[(i, j)]

# gewichtete Kosten (wie im Modell)
total_cost = w_cost * total_distance + w_risk * total_risk

print("\n" + "="*60)
print("ERGEBNISSE")
print("="*60)

print(f"Gesamtdistanz: {total_distance:.2f} km")
print(f"Gesamtrisiko:  {total_risk:.2f}")
print(f"Kostenwert:    {total_cost:.2f}")


ERGEBNISSE
Gesamtdistanz: 14089.90 km
Gesamtrisiko:  1421.50
Kostenwert:    10289.38
